In [ ]:
# Cell 1 — generate short note .wav files (C D E F G) in the current folder
import numpy as np
from scipy.io.wavfile import write

# 5 notes frequencies (C4..G4)
notes = {'C':261.63, 'D':293.66, 'E':329.63, 'F':349.23, 'G':392.00}
sr = 22050
duration = 0.6  # seconds

for name, freq in notes.items():
    t = np.linspace(0, duration, int(sr*duration), False)
    # simple piano-like tone: sine with exponential envelope
    envelope = np.exp(-3 * t)
    tone = 0.5 * np.sin(2*np.pi*freq*t) * envelope
    # normalize to int16
    audio = np.int16(tone/np.max(np.abs(tone)) * 32767)
    write(f"{name}.wav", sr, audio)

print("Generated C.wav, D.wav, E.wav, F.wav, G.wav")


In [1]:
# Cell 2 — hand-gesture piano (run this cell)
import cv2, mediapipe as mp, pygame, time

# Initialize sound system and map counts->wav files (ensure those wavs are in working folder)
pygame.mixer.init()
sounds = {
    1: pygame.mixer.Sound("C.mp3"),
    2: pygame.mixer.Sound("D.wav"),
    3: pygame.mixer.Sound("E.wav"),
    4: pygame.mixer.Sound("F.wav"),
    5: pygame.mixer.Sound("G.mp3")
}

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils
hands = mp_hands.Hands(min_detection_confidence=0.6, min_tracking_confidence=0.6)

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)  # use CAP_DSHOW on Windows if needed
last_count = None
debounce = 0.35
last_time = 0.0

def thumb_open(lm, handedness_label):
    tip_x = lm[4].x; ip_x = lm[3].x
    return (tip_x < ip_x) if handedness_label == 'Right' else (tip_x > ip_x)

try:
    while True:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.flip(frame, 1)
        h, w = frame.shape[:2]
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = hands.process(rgb)

        count = 0
        if res.multi_hand_landmarks:
            lm = res.multi_hand_landmarks[0].landmark
            # tips: index(8), middle(12), ring(16), pinky(20)
            fingers = [1 if lm[i].y < lm[i-2].y else 0 for i in (8,12,16,20)]
            fingers_cnt = sum(fingers)
            handed = 'Right'
            if res.multi_handedness:
                handed = res.multi_handedness[0].classification[0].label
            thumb = 1 if thumb_open(lm, handed) else 0
            count = fingers_cnt + thumb
            mp_draw.draw_landmarks(frame, res.multi_hand_landmarks[0], mp_hands.HAND_CONNECTIONS)

        now = time.time()
        if count != last_count and now - last_time > debounce:
            last_time = now
            last_count = count
            if 1 <= count <= 5:
                sounds[count].play()

        note_text = ("No note" if not (1 <= count <= 5) else f"{['','C','D','E','F','G'][count]}")
        cv2.putText(frame, f"Fingers: {count}   Note: {note_text}", (10,40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2, cv2.LINE_AA)

        cv2.imshow("Hand Piano - press q to quit", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    hands.close()
    pygame.quit()


pygame 2.6.1 (SDL 2.28.4, Python 3.10.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
import cv2
import mediapipe as mp
import pygame
import time

# Initialize pygame mixer and load piano notes
pygame.mixer.init()
sounds = {
    1: pygame.mixer.Sound("C.mp3"),
    2: pygame.mixer.Sound("D.wav"),
    3: pygame.mixer.Sound("E.wav"),
    4: pygame.mixer.Sound("F.wav"),
    5: pygame.mixer.Sound("G.mp3")
}

# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils
hands = mp_hands.Hands(min_detection_confidence=0.6, min_tracking_confidence=0.6)

# Open webcam
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)  # CAP_DSHOW works better on Windows
last_count = None
debounce = 0.35  # seconds between repeated notes
last_time = 0.0

def thumb_open(lm, handedness_label):
    tip_x = lm[4].x
    ip_x = lm[3].x
    return (tip_x < ip_x) if handedness_label == 'Right' else (tip_x > ip_x)

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)  # mirror image
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = hands.process(rgb)

        count = 0
        if res.multi_hand_landmarks:
            lm = res.multi_hand_landmarks[0].landmark
            # Detect fingers: index(8), middle(12), ring(16), pinky(20)
            fingers = [1 if lm[i].y < lm[i-2].y else 0 for i in (8,12,16,20)]
            fingers_cnt = sum(fingers)
            handed = 'Right'
            if res.multi_handedness:
                handed = res.multi_handedness[0].classification[0].label
            thumb = 1 if thumb_open(lm, handed) else 0
            count = fingers_cnt + thumb

            # Draw hand skeleton
            mp_draw.draw_landmarks(frame, res.multi_hand_landmarks[0], mp_hands.HAND_CONNECTIONS)

        # Play note if count changes (debounce)
        now = time.time()
        if count != last_count and now - last_time > debounce:
            last_time = now
            last_count = count
            if 1 <= count <= 5:
                sounds[count].play()

        # Show text: finger count + note
        note_text = "No note" if not (1 <= count <= 5) else f"{['','C','D','E','F','G'][count]}"
        cv2.putText(frame, f"Fingers: {count}   Note: {note_text}", (10,40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2, cv2.LINE_AA)

        # Display live webcam with hand detection
        cv2.imshow("Hand Piano - Press 'q' to quit", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    hands.close()
    pygame.quit()
